In [ ]:
import pandas as pd

# df_raw = pd.read_excel(r'../data/credit_risk_dataset_v2_sample10k.xlsx', sheet_name='Customer Data')
# switched to parquet for faster loading
df_raw = pd.read_parquet(r'../data/credit_risk_dataset_v2.parquet')
print(df_raw.shape)

(10000, 538)


In [ ]:
df_raw.head(5)

,customer_id,risk_attribute_770487,risk_attribute_216739,risk_attribute_126225,risk_attribute_877572,risk_attribute_388389,risk_attribute_356787,risk_attribute_334053,risk_attribute_246316,risk_attribute_872246,...,risk_attribute_815198,risk_attribute_361650,risk_attribute_799330,risk_attribute_207786,risk_attribute_470858,risk_attribute_918011,risk_attribute_687080,risk_attribute_526117,risk_attribute_750810,risk_attribute_885884
0,CUST_0075722,0,0,0,0,0,0,0,0,0,...,3,111192,249174,0.0000,0,6,600,0,61,0
1,CUST_0080185,0,0,0,0,0,0,0,0,0,...,3,71956,140025,0.0000,0,7,600,2,66,1
2,CUST_0019865,0,0,0,0,0,0,0,0,0,...,4,114942,110394,1.6949,0,4,600,1,41,1
3,CUST_0076700,0,0,0,0,0,0,0,0,0,...,4,186962,0,1.8173,0,3,600,1,28,0
4,CUST_0092992,0,0,0,0,0,0,0,0,0,...,2,0,41517,1.2807,0,3,600,0,61,0


In [ ]:
#target columns
# a customer flagged `bad = 1` if any of three risk attributes cross a threshold (one ≥ 8, two others > 0), else `0`
df_raw['bad'] = (
    (df_raw['risk_attribute_383060'] >= 8) |
    (df_raw['risk_attribute_274389'] > 0) |
    (df_raw['risk_attribute_272634'] > 0)
).astype(int)
#  value_counts and mean show the class balance and the resulting bad rate.
print(df_raw['bad'].value_counts())
print("Bad rate:", df_raw['bad'].mean())
# the three columns used to *build* the label are stored in `leakage_cols` and dropped from the features otherwise the model could cheat by learning the exact rule that defined the target. `customer_id` (an identifier, not a predictor) is also dropped.
leakage_cols = ['risk_attribute_383060', 'risk_attribute_274389', 'risk_attribute_272634']
X = df_raw.drop(columns=['customer_id', 'bad'] + leakage_cols)
y = df_raw['bad']
# The result is `X` (features) and `y` (target). The final line confirms there are no missing values.
print("X shape:", X.shape, "y shape:", y.shape)
print("Missing Values in dataset:", X.isnull().sum().sum())

bad
0    9841
1     159
Name: count, dtype: int64
Bad rate: 0.0159
X shape: (10000, 534) y shape: (10000,)
Missing Values in dataset: 0


In [ ]:
#train test split 80/20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(8000, 534) (2000, 534)
0.015875 0.016


In [ ]:
# --- imports + shared setup ---
# The baseline model cells were removed, which took these imports (and `spw`) with them.
# This cell restores what the rest of the notebook depends on.
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV,
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, confusion_matrix,
)

# used by the leakage sweep below
spw = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight (original label):", round(spw, 2))

In [ ]:
# dd = pd.read_excel(r'../data/credit_risk_dataset_v2_sample10k.xlsx', sheet_name='Data Dictionary')
# col2cat = dd.set_index('Column Name')['Category'].to_dict()
# col2def = dd.set_index('Column Name')['Definition'].to_dict()
# print(dd.shape)
# print(dd['Category'].value_counts().head(15).to_string())

dd = pd.read_parquet(r'../data/credit_risk_data_dictionary.parquet')
col2cat = dd.set_index('Column Name')['Category'].to_dict()
col2def = dd.set_index('Column Name')['Definition'].to_dict()

In [ ]:
# What the three label-source columns
for c in leakage_cols:
    print(f"{c}\n  category: {col2cat[c]}\n  def     : {col2def[c]}\n")

c = 'risk_attribute_671412'
print(f"{c}\n  category: {col2cat[c]}\n  def     : {col2def[c]}")

In [ ]:
# how much to remove before the label stops being reconstructible?
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

single_auc = {}
for c in X.columns:
    if X[c].nunique() < 2:
        continue
    a = roc_auc_score(y, X[c])
    single_auc[c] = max(a, 1 - a)
single_auc = pd.Series(single_auc)

rows = []
for thr in [0.95, 0.90, 0.80, 0.70, 0.60, 0.55]:
    drop = single_auc[single_auc > thr].index
    Xd = X.drop(columns=drop)
    a, b, c_, d = train_test_split(Xd, y, test_size=.2, random_state=42, stratify=y)
    mm = XGBClassifier(n_estimators=300, learning_rate=.05, max_depth=4, subsample=.8,
                       colsample_bytree=.8, min_child_weight=5, scale_pos_weight=spw,
                       eval_metric='aucpr', random_state=42, n_jobs=-1).fit(a, c_)
    pp = mm.predict_proba(b)[:, 1]
    rows.append({'drop_above_auc': thr, 'n_dropped': len(drop), 'n_left': Xd.shape[1],
                 'roc_auc': roc_auc_score(d, pp), 'pr_auc': average_precision_score(d, pp)})

print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
DEROGATORY_CATEGORIES = {
    'Delinquency Percentages',
    'Delinquency Counts – 30 Days',
    'Delinquency Counts – 60 Days and Severe',
    'Worst Status Codes',
    'Collections and Derogatory Metrics',
    'Months Since Delinquency',
    'Public Records and Bankruptcy',
    'Additional Delinquency Severity and Recurrence',
    'Derived and Composite Risk Indicators',
    'Financial Stress and Hardship Indicators',
    'Payment History and Behavior',
}

y_v2 = (df_raw['risk_attribute_272634'] > 0).astype(int)
clean_cols = [c for c in df_raw.columns
              if c != 'customer_id' and col2cat.get(c) not in DEROGATORY_CATEGORIES]
X_v2 = df_raw[clean_cols]

print("features:", X_v2.shape[1], "(was 534)")
print("positives:", int(y_v2.sum()), " rate:", round(y_v2.mean(), 4))
print("\nsurviving categories:")
print(pd.Series([col2cat.get(c) for c in clean_cols]).value_counts().to_string())

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)
spw_v2 = (y2_train == 0).sum() / (y2_train == 1).sum()

xgb_v2 = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.6,
    min_child_weight=10,
    reg_lambda=5.0,
    scale_pos_weight=spw_v2,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)
xgb_v2.fit(X2_train, y2_train)

proba_v2 = xgb_v2.predict_proba(X2_test)[:, 1]
pr = average_precision_score(y2_test, proba_v2)
floor = y2_test.mean()

print("ROC-AUC : %.4f" % roc_auc_score(y2_test, proba_v2))
print("PR-AUC  : %.4f" % pr)
print("Floor   : %.4f  (random guessing)" % floor)
print("Lift    : %.1fx over random" % (pr / floor))

In [ ]:
# Cross-validated only around 15 positives land in a single test split, so one number is unreliable.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_v2 = cross_val_score(xgb_v2, X_v2, y_v2, cv=cv, scoring="average_precision", n_jobs=-1)

print("Fold PR-AUCs:", np.round(cv_v2, 4))
print("Mean: %.4f  (+/- %.4f)" % (cv_v2.mean(), cv_v2.std()))

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

prec, rec, thresh = precision_recall_curve(y2_test, proba_v2)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = f1.argmax()

print("Best-F1 threshold: %.4f" % thresh[min(best, len(thresh) - 1)])
print("  precision %.3f   recall %.3f   F1 %.3f" % (prec[best], rec[best], f1[best]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec, prec, lw=2)
ax.axhline(floor, ls="--", c="grey", label="random (%.4f)" % floor)
ax.scatter(rec[best], prec[best], c="red", zorder=5, label="best F1")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("v2 — bankruptcy, non-derogatory features")
ax.legend(); plt.show()

pred_v2 = (proba_v2 >= thresh[min(best, len(thresh) - 1)]).astype(int)
print(confusion_matrix(y2_test, pred_v2))
print(classification_report(y2_test, pred_v2, digits=3))

In [ ]:
imp_v2 = pd.Series(xgb_v2.feature_importances_, index=X_v2.columns).sort_values(ascending=False)

top = pd.DataFrame({
    "gain": imp_v2.head(15).round(4),
    "category": [col2cat.get(c) for c in imp_v2.head(15).index],
    "definition": [str(col2def.get(c))[:70] for c in imp_v2.head(15).index],
})
print("top-1 share of gain: %.1f%%  (baseline was 37.7%%)" % (100 * imp_v2.iloc[0] / imp_v2.sum()))
print(top.to_string())

In [ ]:
#hyperparameter search
from scipy.stats import loguniform, randint, uniform

USE_GPU = False   # set True if you have an NVIDIA GPU; see note below
N_CANDIDATES = 100
N_JOBS = 4        # set to your physical core count, not -1

base = XGBClassifier(
    eval_metric="aucpr",
    random_state=42,
    tree_method="hist",
    device="cuda" if USE_GPU else "cpu",
    n_jobs=1,     # parallelism comes from the search, not the fit
)

param_dist = {
    "n_estimators":     randint(200, 1200),
    "learning_rate":    loguniform(0.01, 0.2),
    "max_depth":        randint(2, 7),
    "min_child_weight": randint(1, 30),
    "subsample":        uniform(0.6, 0.4),      # 0.6 - 1.0
    "colsample_bytree": uniform(0.3, 0.7),      # 0.3 - 1.0
    "reg_lambda":       loguniform(0.1, 50.0),
    "reg_alpha":        loguniform(1e-3, 10.0),
    "gamma":            loguniform(1e-4, 5.0),
    # searched, not fixed at n_neg/n_pos
    "scale_pos_weight": loguniform(1.0, spw_v2 * 2),
}

search = RandomizedSearchCV(
    base,
    param_distributions=param_dist,
    n_iter=N_CANDIDATES,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=69),
    n_jobs=N_JOBS,
    refit=True,
    random_state=69,
    verbose=1,
    return_train_score=False,
)

search.fit(X2_train, y2_train)   # test set deliberately untouched

print("\nbest CV PR-AUC: %.4f" % search.best_score_)
for k, v in sorted(search.best_params_.items()):
    print(f"  {k:18} {v:.4f}" if isinstance(v, float) else f"  {k:18} {v}")

In [ ]:
xgb_tuned = search.best_estimator_
proba_tuned = xgb_tuned.predict_proba(X2_test)[:, 1]

pr_v2    = average_precision_score(y2_test, proba_v2)
pr_tuned = average_precision_score(y2_test, proba_tuned)
floor    = y2_test.mean()

results = pd.DataFrame([
    {"model": "v2 (hand-tuned)", "cv_pr_auc": cv_v2.mean(),
     "heldout_pr_auc": pr_v2,    "heldout_roc_auc": roc_auc_score(y2_test, proba_v2)},
    {"model": "v2 (searched)",   "cv_pr_auc": search.best_score_,
     "heldout_pr_auc": pr_tuned, "heldout_roc_auc": roc_auc_score(y2_test, proba_tuned)},
])
results["lift_vs_random"] = (results["heldout_pr_auc"] / floor).round(1)
results["cv_minus_heldout"] = (results["cv_pr_auc"] - results["heldout_pr_auc"]).round(4)

print("random-guessing floor: %.4f\n" % floor)
print(results.round(4).to_string(index=False))

gap = search.best_score_ - pr_tuned
print("\nwinner's-curse gap: %+.4f" % gap)
if gap > 2 * cv_v2.std():
    print("CV score is optimistic by more than 2 CV std devs.")
    print("Report the held-out number, not best_score_.")
if pr_tuned <= pr_v2:
    print("Search did NOT beat the hand-tuned model on held-out data.")
    print("Expected on the 10k sample. Keep the simpler model.")

In [ ]:
#search surface flatness
cvres = pd.DataFrame(search.cv_results_)
top = cvres.nsmallest(20, "rank_test_score")[
    ["rank_test_score", "mean_test_score", "std_test_score",
     "param_max_depth", "param_min_child_weight", "param_scale_pos_weight",
     "param_learning_rate", "param_reg_lambda"]
]
print(top.round(4).to_string(index=False))

spread = top["mean_test_score"].max() - top["mean_test_score"].min()
print("\nspread across top-20: %.4f   typical fold std: %.4f"
      % (spread, top["std_test_score"].mean()))
if spread < top["std_test_score"].mean():
    print("Top-20 are within noise of each other; ranking not meaningful.")